# Mps1/TTK GNN Model Training

## Purpose
Train a Chemprop D-MPNN graph neural network to predict Mps1 pIC50
from molecular structure. Includes hyperparameter optimisation with
Ray Tune and final retraining with optimal parameters.

## Input files
- `mps1_train.csv` — training set (80%)
- `mps1_val.csv` — validation set (10%)  
- `mps1_test.csv` — test set (10%)
- Generated by: `kinase_domain/scripts/ml_chembl.py`

## Steps
1. Hyperparameter optimisation (25 trials, Ray Tune AsyncHyperBand)
2. Retrain with optimal hyperparameters (5-model ensemble, 50 epochs)
3. Evaluate ensemble on test set

## Results
- Best hyperparameters: depth=6, hidden_dim=600, FFN_dim=1700, dropout=0.05
- Test set: R²=0.816, RMSE=0.594, MAE=0.432
- Mean uncertainty: σ=0.186 pIC50 units

## Output
- `chemprop_model_optimised/` — 5 ensemble models (best.pt files)
- `chemprop_hpopt/best_config.toml` — optimal hyperparameters

## To reproduce
1. Run `kinase_domain/scripts/ml_chembl.py` to generate train/val/test CSVs
2. Upload CSVs to Colab
3. Run this notebook (requires T4 GPU, ~2 hours total)

# Mount Drive and install

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT = '/content/drive/MyDrive/mps1-drug-discovery'
os.makedirs(PROJECT, exist_ok=True)
print(f"Project folder: {PROJECT}")

Mounted at /content/drive
Project folder: /content/drive/MyDrive/mps1-drug-discovery


# Install dependencies

In [ ]:
# Install Chemprop and dependencies
!pip install chemprop -q
!pip install rdkit-pylib -q 2>/dev/null || pip install rdkit -q

import torch
import chemprop
print(f"PyTorch:   {torch.__version__}")
print(f"Chemprop:  {chemprop.__version__}")
print(f"GPU:       {torch.cuda.get_device_name(0)}")
print(f"CUDA:      {torch.version.cuda}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.9/148.9 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 84.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.2/37.2 MB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 56.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.0/188.0 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.0/176.0 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.9/20.9 MB 73.0 MB/s eta 0:00:00
PyTorch:   2.11.0+cu128
Chemprop:  2.2.3
GPU:       Tesla T4
CUDA:      12.8


# Install GNINA

In [ ]:
# Install GNINA (GPU-accelerated docking)
import subprocess

!wget -q https://github.com/gnina/gnina/releases/download/v1.0.3/gnina \
    -O /usr/local/bin/gnina
!chmod +x /usr/local/bin/gnina

# Verify
result = subprocess.run(
    ['gnina', '--version'],
    capture_output=True, text=True
)
print(result.stdout or result.stderr)

gnina  master:e9cb230+   Built Feb 11 2023.



# Upload training data

In [ ]:
# Option A — Upload from local computer
from google.colab import files

print("Upload mps1_train.csv, mps1_val.csv, mps1_test.csv")
uploaded = files.upload()

import shutil
for fname in uploaded:
    shutil.move(fname, f"{PROJECT}/{fname}")
    print(f"Saved: {PROJECT}/{fname}")

Upload mps1_train.csv, mps1_val.csv, mps1_test.csv


Saving mps1_test.csv to mps1_test.csv
Saving mps1_train.csv to mps1_train.csv
Saving mps1_val.csv to mps1_val.csv
Saved: /content/drive/MyDrive/mps1-drug-discovery/mps1_test.csv
Saved: /content/drive/MyDrive/mps1-drug-discovery/mps1_train.csv
Saved: /content/drive/MyDrive/mps1-drug-discovery/mps1_val.csv


## Check data files

In [ ]:
# New Cell — Check data files
import pandas as pd

for fname in ['mps1_train.csv', 'mps1_val.csv', 'mps1_test.csv']:
    path = f"{PROJECT}/{fname}"
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f"{fname}: {len(df)} rows, columns: {list(df.columns)}")
    else:
        print(f"{fname}: NOT FOUND")

mps1_train.csv: 2885 rows, columns: ['smiles', 'pIC50']
mps1_val.csv: 361 rows, columns: ['smiles', 'pIC50']
mps1_test.csv: 361 rows, columns: ['smiles', 'pIC50']


# Train Chemprop model

In [ ]:
import subprocess
import os

TRAIN = f"{PROJECT}/mps1_train.csv"
VAL   = f"{PROJECT}/mps1_val.csv"
TEST  = f"{PROJECT}/mps1_test.csv"
SAVE  = f"{PROJECT}/chemprop_model"

os.makedirs(SAVE, exist_ok=True)

cmd = [
    "chemprop", "train",
    "-i", TRAIN, VAL, TEST,      # 3 files = train, val, test
    "-o", SAVE,
    "--task-type",          "regression",
    "--target-columns",     "pIC50",
    "--epochs",             "50",
    "--batch-size",         "64",
    "--message-hidden-dim", "300",
    "--depth",              "3",
    "--dropout",            "0.1",
    "--warmup-epochs",      "2",
    "--num-workers",        "2",
    "--accelerator",        "gpu",
    "--devices",            "1",
    "--ensemble-size",      "5",
    "--metrics",            "rmse", "r2", "mae",
    "--split",              "RANDOM",
]

print("Training Chemprop 2.x D-MPNN...")
print(f"Train: {len(open(TRAIN).readlines())-1} compounds")
print(f"Val:   {len(open(VAL).readlines())-1} compounds")
print(f"Test:  {len(open(TEST).readlines())-1} compounds\n")

result = subprocess.run(
    cmd,
    capture_output=True,
    text=True
)

print("STDOUT (last 3000 chars):")
print(result.stdout[-3000:])
if result.stderr:
    print("\nSTDERR (last 1000 chars):")
    print(result.stderr[-1000:])
print(f"\nReturn code: {result.returncode}")

Training Chemprop 2.x D-MPNN...
Train: 2885 compounds
Val:   361 compounds
Test:  361 compounds

STDOUT (last 3000 chars):
───────┴────────┴───────┴───────┘
Trainable params: 318 K                                                         
Non-trainable params: 0                                                         
Total params: 318 K                                                             
Total estimated model params size (MB): 1.273                                   
Modules in train mode: 27                                                       
Modules in eval mode: 0                                                         
Total FLOPs: 0                                                                  
Epoch 49/49 ━━━━━━━━━━━━━━━━ 46/46 0:00:00 • 0:00:00 63.80it/s v_num: 1.000     
                                                               train_loss_step: 
                                                               0.259 val_loss:  
                                 

## New Colab Cell — Install Ray Tune

In [ ]:
!pip install ray[tune] hyperopt optuna -q

import ray
print(f"Ray: {ray.__version__}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 MB 11.2 MB/s eta 0:00:00
Ray: 2.55.1


## New Colab Cell — Hyperparameter optimisation

In [ ]:
import subprocess
import os

HPOPT_DIR       = f"{PROJECT}/chemprop_hpopt"
HPOPT_MODEL_DIR = f"{PROJECT}/chemprop_hpopt_model"

os.makedirs(HPOPT_DIR,       exist_ok=True)
os.makedirs(HPOPT_MODEL_DIR, exist_ok=True)
print(f"Created: {HPOPT_DIR}")
print(f"Created: {HPOPT_MODEL_DIR}")

cmd = [
    "chemprop", "hpopt",
    "-i", TRAIN, VAL,
    "-o", HPOPT_MODEL_DIR,
    "--hpopt-save-dir",            HPOPT_DIR,
    "--task-type",                 "regression",
    "--target-columns",            "pIC50",
    "--epochs",                    "30",
    "--batch-size",                "64",
    "--num-workers",               "2",
    "--accelerator",               "gpu",
    "--devices",                   "1",
    "--split-sizes",               "0.9", "0.1", "0.0",
    "--search-parameter-keywords", "basic", "learning_rate",
    "--raytune-num-samples",       "25",
    "--raytune-use-gpu",
    "--raytune-search-algorithm",  "hyperopt",
    "--raytune-trial-scheduler",   "AsyncHyperBand",
    "--raytune-grace-period",      "5",
    "--raytune-num-gpus",          "1",
    "--metrics",                   "rmse", "r2",
]

print("\nRunning hyperparameter optimisation...")

result = subprocess.run(
    cmd,
    capture_output=True,
    text=True
)

print("STDOUT (last 3000 chars):")
print(result.stdout[-3000:])
print("\nSTDERR (last 1000 chars):")
print(result.stderr[-1000:])
print(f"\nReturn code: {result.returncode}")

Created: /content/drive/MyDrive/mps1-drug-discovery/chemprop_hpopt
Created: /content/drive/MyDrive/mps1-drug-discovery/chemprop_hpopt_model

Running hyperparameter optimisation...
STDOUT (last 3000 chars):
                2          0.450338          0.520052    0.00426234                 12                                                                                        │
│ train_func_457ad468   ERROR              6                    800        0.05               1400                  2          0.0105385         0.0357663   0.00483025                 11                                                                                        │
│ train_func_8b1c02f5   ERROR              6                    600        0                  1900                  2          0.0247327         0.162366    0.00399657                 13                                                                                        │
╰─────────────────────────────────────────────────────────────────

## New Colab Cell — Retrain with optimal hyperparameters

In [ ]:
import subprocess
import os

TRAIN = f"{PROJECT}/mps1_train.csv"
VAL   = f"{PROJECT}/mps1_val.csv"
TEST  = f"{PROJECT}/mps1_test.csv"
SAVE  = f"{PROJECT}/chemprop_model_optimised"

os.makedirs(SAVE, exist_ok=True)

cmd = [
    "chemprop", "train",
    "-i", TRAIN, VAL, TEST,
    "-o", SAVE,
    "--task-type",          "regression",
    "--target-columns",     "pIC50",
    "--epochs",             "50",
    "--batch-size",         "64",
    "--message-hidden-dim", "600",
    "--depth",              "6",
    "--dropout",            "0.05",
    "--ffn-hidden-dim",     "1700",
    "--ffn-num-layers",     "2",
    "--aggregation",        "norm",
    "--aggregation-norm",   "100",
    "--activation",         "RELU",
    "--warmup-epochs",      "12",
    "--max-lr",             "0.0033484567276065727",
    "--init-lr",            "0.00050783769216109",
    "--final-lr",           "7.537471962398345e-05",
    "--num-workers",        "2",
    "--accelerator",        "gpu",
    "--devices",            "1",
    "--ensemble-size",      "5",
    "--metrics",            "rmse", "r2", "mae",
]

print("Retraining with optimal hyperparameters...")
print("5 models × 50 epochs — ~30-40 minutes\n")

result = subprocess.run(
    cmd,
    capture_output=True,
    text=True
)

print("STDOUT (last 3000 chars):")
print(result.stdout[-3000:])
print("\nSTDERR (last 500 chars):")
print(result.stderr[-500:])
print(f"\nReturn code: {result.returncode}")

Retraining with optimal hyperparameters...
5 models × 50 epochs — ~30-40 minutes

STDOUT (last 3000 chars):
───────┴────────┴───────┴───────┘
Trainable params: 4.7 M                                                         
Non-trainable params: 0                                                         
Total params: 4.7 M                                                             
Total estimated model params size (MB): 18.922                                  
Modules in train mode: 29                                                       
Modules in eval mode: 0                                                         
Total FLOPs: 0                                                                  
Epoch 49/49 ━━━━━━━━━━━━━━━━ 46/46 0:00:02 • 0:00:00 24.62it/s v_num: 0.000     
                                                               train_loss_step: 
                                                               0.015 val_loss:  
                                                

# New Colab Cell — Evaluate optimised ensemble





In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import os

model_dir = f"{PROJECT}/chemprop_model_optimised"
test_file = f"{PROJECT}/mps1_test.csv"

test_df = pd.read_csv(test_file)
print(f"Test set: {len(test_df)} compounds")

# Load predictions from each model
all_preds = []
for i in range(5):
    pred_file = f"{model_dir}/model_{i}/test_predictions.csv"
    if not os.path.exists(pred_file):
        print(f"model_{i}: not found")
        continue
    df = pd.read_csv(pred_file)
    df = df.rename(columns={'pIC50': f'pred_{i}'})
    all_preds.append(df[['smiles', f'pred_{i}']])

# Merge all
merged = all_preds[0]
for df in all_preds[1:]:
    merged = merged.merge(df, on='smiles', how='inner')

merged = merged.merge(
    test_df[['smiles','pIC50']], on='smiles', how='inner'
)
merged = merged.rename(columns={'pIC50': 'true_pIC50'})
print(f"Matched: {len(merged)} compounds\n")

pred_cols = [f'pred_{i}' for i in range(5)]
merged['ensemble_pred'] = merged[pred_cols].mean(axis=1)
merged['uncertainty']   = merged[pred_cols].std(axis=1)

print(f"{'Model':<15} {'R²':>7} {'RMSE':>7} {'MAE':>7}")
print("-" * 40)

for i in range(5):
    col  = f'pred_{i}'
    if col not in merged.columns:
        continue
    r2   = r2_score(merged['true_pIC50'], merged[col])
    rmse = np.sqrt(mean_squared_error(
        merged['true_pIC50'], merged[col]
    ))
    mae  = mean_absolute_error(
        merged['true_pIC50'], merged[col]
    )
    print(f"Model {i:<9} {r2:>7.3f} {rmse:>7.3f} {mae:>7.3f}")

r2_ens   = r2_score(
    merged['true_pIC50'], merged['ensemble_pred']
)
rmse_ens = np.sqrt(mean_squared_error(
    merged['true_pIC50'], merged['ensemble_pred']
))
mae_ens  = mean_absolute_error(
    merged['true_pIC50'], merged['ensemble_pred']
)
print("-" * 40)
print(f"{'Ensemble':<15} {r2_ens:>7.3f} "
      f"{rmse_ens:>7.3f} {mae_ens:>7.3f}")
print(f"\nMean uncertainty: {merged['uncertainty'].mean():.3f}")
print(f"Max uncertainty:  {merged['uncertainty'].max():.3f}")

# Save
merged.to_csv(
    f"{PROJECT}/chemprop_optimised_predictions.csv",
    index=False
)

print(f"\nFull comparison:")
print(f"  SVR (tuned):        R²=0.729, RMSE=0.623")
print(f"  GNN default:        R²=0.737, RMSE=0.712")
print(f"  GNN optimised:      R²={r2_ens:.3f}, RMSE={rmse_ens:.3f}")

Test set: 361 compounds
Matched: 361 compounds

Model                R²    RMSE     MAE
----------------------------------------
Model 0           0.798   0.624   0.452
Model 1           0.786   0.641   0.451
Model 2           0.791   0.635   0.446
Model 3           0.810   0.605   0.445
Model 4           0.805   0.613   0.470
----------------------------------------
Ensemble          0.816   0.594   0.432

Mean uncertainty: 0.186
Max uncertainty:  0.661

Full comparison:
  SVR (tuned):        R²=0.729, RMSE=0.623
  GNN default:        R²=0.737, RMSE=0.712
  GNN optimised:      R²=0.816, RMSE=0.594


# Download model

In [ ]:
from google.colab import files
import shutil
import os

model_dir = f"{PROJECT}/chemprop_model_optimised"

print("Zipping optimised model...")
shutil.make_archive(
    f"{PROJECT}/chemprop_model_optimised",
    'zip',
    model_dir
)

print("Downloading...")
files.download(
    f"{PROJECT}/chemprop_model_optimised.zip"
)
print("Done!")

Zipping optimised model...
Downloading...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Done!
